# Seed HR

Resets the HR harness, then adds a small Faker load through `POST /admin/seed`.
The service generates the rows and writes them with SQLAlchemy. This notebook
does not open the SQLite file.

The default is 200 people on top of the curated seed (48 people and 38 policies).
`SEED_COUNT=0` skips the extra rows. `SEED_COUNT=10000` is the scale load.
`SEED_RESET=false` keeps rows that are already there. `SEED_KIND=policy` loads
policies instead of people.


## Parameters


In [ ]:
import os
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "hr_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import hr_demo

SOURCE = os.environ.get("SEED_SOURCE", "hr")
KIND = os.environ.get("SEED_KIND", "person")
COUNT = int(os.environ.get("SEED_COUNT", "200"))
SEED = int(os.environ.get("SEED_VALUE", "1"))
RESET = os.environ.get("SEED_RESET", "true").lower() == "true"
if SOURCE != "hr":
    raise SystemExit(f"{SOURCE} has no seed endpoint yet")
print(SOURCE, KIND, COUNT, SEED, RESET)


## Load


In [ ]:
if RESET:
    print(hr_demo.admin("POST", "/admin/reset", timeout=180))
if COUNT:
    result = hr_demo.admin(
        "POST",
        "/admin/seed",
        {"kind": KIND, "count": COUNT, "seed": SEED, "replace": False},
        timeout=600,
    )
    elapsed = result["elapsed_seconds"] or 0
    rate = result["inserted"] / elapsed if elapsed else 0
    print(result)
    print(f"{result['inserted']} {KIND} rows in {elapsed}s ({rate:.0f} rows/s)")
else:
    print("skipping the Faker top-up")


## Check the load


In [ ]:
discover = hr_demo.admin("GET", "/discover")
ids = hr_demo.admin("GET", "/ids")
print("counts", discover["counts"])
print("ids", {kind: len(rows) for kind, rows in ids["ids"].items()})
assert discover["counts"][KIND] == len(ids["ids"][KIND])
